# Data Preprocessing & Pipeline Construction

## 1. Mục Tiêu Notebook
- Đọc dataset raw và áp dụng kết quả từ EDA.
- Chuẩn hóa tên cột sang tên field chuẩn nội bộ (`snake_case`).
- Loại bỏ các bản ghi trùng lặp và các cột rò rỉ dữ liệu (Data Leakage).
- Thực hiện Train/Test Split (80/20) với `random_state=42` **trước khi fit preprocessor** để triệt tiêu nguy cơ Data Leakage.
- Xây dựng scikit-learn `ColumnTransformer` cho cả thuộc tính số (`Numeric`) và thuộc tính phân loại (`Categorical`).
- Kiểm tra biến đổi trên `X_train` và `X_test` để sẵn sàng cho Giai đoạn 2 (Model Training).

## 2. Import Thư Viện & Module Nội Bộ

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Xác định root project và thêm ai-models vào sys.path
current_dir = Path.cwd()
project_root = current_dir.parent.parent if current_dir.name == 'colab' else current_dir
ai_models_dir = project_root / 'ai-models'
if str(ai_models_dir) not in sys.path:
    sys.path.insert(0, str(ai_models_dir))

from src.preprocess import (
    load_data,
    rename_columns,
    clean_data,
    select_features,
    split_data,
    build_preprocessor,
    CANONICAL_FEATURES,
    TARGET_NAME,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

print('Import các module từ src.preprocess thành công!')

## 3. Load & Chuẩn Hóa Dữ Liệu Raw

In [ ]:
# 1. Load data
df_raw = load_data()
print(f'Raw dataset shape: {df_raw.shape}')

# 2. Rename columns sang tên field chuẩn nội bộ
df_renamed = rename_columns(df_raw)
print('\nDanh sách các cột sau khi chuẩn hóa tên:')
print(df_renamed.columns.tolist())

# 3. Clean data (chuyển chữ hoa + xóa duplicate)
df_cleaned = clean_data(df_renamed)
print(f'\nShape sau khi làm sạch & bỏ duplicate: {df_cleaned.shape}')

## 4. Tách Tập Feature (X) và Target (y)

Chỉ giữ 7 canonical features và 1 canonical target `fuel_consumption_comb`. Loại bỏ hoàn toàn các cột rò rỉ dữ liệu.

In [ ]:
X, y = select_features(df_cleaned)
print('=== TẬP FEATURE (X) ===')
print('Shape X:', X.shape)
print('Cột X:', X.columns.tolist())

print('\n=== TẬP TARGET (y) ===')
print('Shape y:', y.shape)
print('Tên target:', y.name)

print('\n5 Dòng đầu tiên của X:')
display(X.head())

## 5. Train / Test Split (Ngừa Data Leakage)

Phân chia tập dữ liệu thành 80% Train và 20% Test với `random_state=42` **TRƯỚC KHI** fit bất kỳ bộ chuyển đổi nào.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print('=== KẾT QUẢ TRAIN/TEST SPLIT ===')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

assert len(X_train) + len(X_test) == len(X), 'Tổng số dòng split không khớp!'

## 6. Xây Dựng Preprocessing Pipeline (ColumnTransformer)

Xây dựng Pipeline độc lập:
- **Numeric Features** (`model_year`, `engine_size`, `cylinders`): `SimpleImputer(strategy='median')` -> `StandardScaler()`
- **Categorical Features** (`make`, `vehicle_class`, `transmission`, `fuel_type`): `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
preprocessor = build_preprocessor()
print('Cấu trúc ColumnTransformer:')
print(preprocessor)

## 7. Fit & Transform Dữ Liệu Train / Test

**BẮT BUỘC**: Fit preprocessor CHỈ TRÊN `X_train`. Không fit trên `X_test` hoặc toàn bộ `X`.

In [ ]:
# Fit preprocessor chỉ trên X_train
preprocessor.fit(X_train)

# Transform X_train và X_test
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print('=== KẾT QUẢ BIẾN ĐỔI PREPROCESSING ===')
print(f'X_train biến đổi shape: {X_train_transformed.shape}')
print(f'X_test biến đổi shape:  {X_test_transformed.shape}')

# Kiểm tra không có NaN bất thường
has_nan_train = np.isnan(X_train_transformed.data if hasattr(X_train_transformed, 'data') else X_train_transformed).any()
has_nan_test = np.isnan(X_test_transformed.data if hasattr(X_test_transformed, 'data') else X_test_transformed).any()
print(f'X_train_transformed có NaN không?: {has_nan_train}')
print(f'X_test_transformed có NaN không?:  {has_nan_test}')

## 8. Kết Luận Giai Đoạn Preprocessing
- Preprocessing pipeline hoàn thành xuất sắc và sẵn sàng tích hợp trực tiếp vào giai đoạn train mô hình.
- Đã triệt tiêu rò rỉ dữ liệu thông qua quy trình Split trước Fit.
- Schema định dạng `schema.json` đã được đồng bộ chuẩn hóa để chuyển sang cho Backend/Frontend.